# components/step_renderer

> Main step renderer combining all panels for the source selection step

In [ ]:
#| default_exp components.step_renderer

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Any, Dict, List, Optional

from fasthtml.common import Div, H2, P, Script

from cjm_fasthtml_file_browser.core.config import FileBrowserConfig
from cjm_fasthtml_file_browser.core.models import BrowserState
from cjm_fasthtml_file_browser.providers.local import LocalFileSystemProvider

from cjm_fasthtml_daisyui.utilities.semantic_colors import text_dui

from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w, h
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, flex_direction, grid_display, grid_cols, gap
)
from cjm_fasthtml_tailwind.core.base import combine_classes

from cjm_transcription_source_select.models import (
    SourceSelectUrls, SelectedFile, ExtractionResult
)
from cjm_transcription_source_select.html_ids import SourceSelectHtmlIds
from cjm_transcription_source_select.components.file_browser_panel import (
    get_browser_state, sync_browser_selection, render_browser_panel
)
from cjm_transcription_source_select.components.selection_panel import render_selection_panel
from cjm_transcription_source_select.components.preview_panel import render_preview_panel
from cjm_transcription_source_select.components.stats_panel import render_stats_panel
from cjm_transcription_source_select.components.helpers import generate_sortable_init_script

## Step Renderer

Composes all panels into a single step view:
- Two-column layout: file browser (left) + selection queue (right)
- Collapsible preview panel below
- Stats / verify panel below
- SortableJS scripts at the bottom

In [ ]:
#| export
def render_source_select_step(
    selected_files: List[SelectedFile],  # Ordered selection
    extraction_results: Dict[str, ExtractionResult],  # video_path -> result
    verified: bool,  # Whether selection is verified
    urls: SourceSelectUrls,  # URL bundle
    provider: LocalFileSystemProvider,  # File system provider
    browser_config: FileBrowserConfig,  # Browser configuration
    home_path: str,  # Home directory path
    browser_state: Optional[BrowserState] = None,  # Browser state (created if None)
    start_path: str = "",  # Starting directory (used if no browser_state)
) -> Div:  # Complete step view
    """Render the complete source selection step."""
    # Get or create browser state
    if browser_state is None:
        browser_state = BrowserState(current_path=start_path or home_path)

    # Sync browser selection highlights
    sync_browser_selection(browser_state, selected_files)

    # Render panels
    browser_panel = render_browser_panel(
        browser_state=browser_state,
        config=browser_config,
        provider=provider,
        navigate_url=urls.navigate,
        select_url=urls.select,
        home_path=home_path,
    )

    selection_panel = render_selection_panel(selected_files, urls, extraction_results)
    preview_panel = render_preview_panel(media_src_url=urls.media_src)
    stats_panel = render_stats_panel(selected_files, urls, extraction_results, verified)

    return Div(
        # Two-column layout (browser left, selection right)
        Div(
            Div(browser_panel, cls=w.full),
            Div(selection_panel, cls=w.full),
            cls=combine_classes(str(grid_display), grid_cols(1), grid_cols(2).lg, gap(4))
        ),

        # Preview panel
        preview_panel,

        # Stats / verify panel
        stats_panel,

        # SortableJS library + initialization
        Script(src="https://cdn.jsdelivr.net/npm/sortablejs@1.15.0/Sortable.min.js"),
        Script(generate_sortable_init_script()),

        # Script runner for OOB-triggered JS
        Div(id=SourceSelectHtmlIds.SCRIPT_RUNNER),

        id=SourceSelectHtmlIds.MAIN_CONTAINER,
        cls=combine_classes(w.full, str(flex_display), flex_direction.col, p(4))
    )

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()